# Contribute to KataGo training on Modal

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SoumyaK4/SWHub/blob/SWHub/Docs/Free%20Cloud%20Katago/Contribute/Kata_Modal_Contribution.ipynb)

Use the **control panel** below. You can choose settings, start the server, copy its connection address and stop it without editing code.

During setup, the panel shows the current step, elapsed time and three recent log lines. If a step is quiet, it shows the time since the last log. Full logs remain below.

## Open the panel

1. Open this notebook in **Colab with a CPU runtime**, or in local Jupyter. Modal supplies the T4 GPU. Have a Modal account with a valid payment method, even when using free credits.
2. Click **▶ on the Open control panel block directly below this guide**. Wait for the panel to appear in that block. This is the only code block you need to run. It prepares the controls without logging in, starting a server, deploying an app or running Stop.
3. In the panel, click **1. Login**, open the printed browser link and authorize your intended workspace. Choose it from the dropdown, leave Environment as `main`, then click **2. Select workspace**. The selection is checked without starting a GPU.
4. Leave the optional Controller login disabled for your first setup. Click **3. Deploy**. Wait for the action to finish.
5. Copy the complete address in **SWHub URL** into **Tools > KataGo Contribution > Settings**. Enter your verified [katagotraining.org](https://katagotraining.org/accounts/signup/) account in SWHub, then select **Start contributing**.
6. Select **Finish games & stop** in SWHub and wait for the games to upload. Then click **Stop Modal app** in this panel. Force stop and disconnects can abandon unfinished games.

Keep the connection address private. You can close Colab after deployment. Closing the notebook does not stop the deployed Modal app. Stopping keeps cached downloads. To use it again, click Deploy.

All code blocks are collapsed by default. Expand them only if you want to inspect the implementation. If controls do not appear, reconnect and run the top Open control panel block again. Trust the notebook in local Jupyter. After changing settings, stop an active Colab server first, then start again. Modal changes take effect after Deploy.


In [ ]:
# @title ▶ Open control panel
# Generated by tool/katago_notebooks/sync_controls.py.
# Implementation sources remain readable in the collapsed cells below.
%pip install -q modal==1.6.3 "ipywidgets>=7.7.1,<9"

try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()

"""Shared controls embedded in all four standalone KataGo notebooks.

Run sync_controls.py after editing. No repository checkout is needed by users.
"""

import ast
import builtins
from collections import deque
import html
import json
import re
import subprocess
import threading
import time
from pathlib import Path

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import display


# Registering a cell saves its source without running setup or starting a GPU.
SWHUB_ACTIONS = {}


def swhub_action(line, cell):
    SWHUB_ACTIONS[line.strip()] = cell


get_ipython().register_magic_function(swhub_action, "cell", "swhub_action")


def swhub_configure_source(source, settings):
    """Replace only top-level setting assignments, preserving embedded servers."""
    tree = ast.parse(source)
    lines = source.splitlines(keepends=True)
    for node in reversed(tree.body):
        if not isinstance(node, ast.Assign) or len(node.targets) != 1:
            continue
        target = node.targets[0]
        if isinstance(target, ast.Name) and target.id in settings:
            lines[node.lineno - 1:node.end_lineno] = [
                f"{target.id} = {settings[target.id]!r}\n"
            ]
    return "".join(lines)


def swhub_run_action(name, settings=None, on_output=None):
    source = SWHUB_ACTIONS[name]
    # Existing writefile cells remain readable, self-contained service sources.
    lines = source.splitlines(keepends=True)
    writefile = next((i for i, line in enumerate(lines)
                      if line.startswith("%%writefile ")), None)
    if writefile is not None:
        path = Path(lines[writefile].split(maxsplit=1)[1].strip())
        source = swhub_configure_source("".join(lines[writefile + 1:]), settings or {})
        compile(source, str(path), "exec")
        path.write_text(source, encoding="utf-8")
        print("Prepared", path.name)
        if on_output is not None:
            on_output("Prepared " + path.name)
        return
    source = swhub_configure_source(source, settings or {})
    # Observe Python action logs without redirecting the kernel's stdout or
    # moving Drive authorization and IPython work off the calling thread.
    had_print = "print" in globals()
    original_print = globals().get("print", builtins.print)

    def report(*args, **kwargs):
        original_print(*args, **kwargs)
        if on_output is not None and kwargs.get("file") is None:
            separator = kwargs.get("sep", " ")
            on_output((" " if separator is None else separator).join(str(arg) for arg in args))

    globals()["print"] = report
    try:
        exec(compile(source, f"KataGo action: {name}", "exec"), globals())
    finally:
        if had_print:
            globals()["print"] = original_print
        else:
            globals().pop("print", None)


class SWHubKataGoPanel:
    def __init__(self, kind):
        self.kind = kind
        self.modal = kind.startswith("modal")
        self.contribution = kind.endswith("contribution")
        self.busy = False
        self.target = None
        self.activity_stop = threading.Event()
        self.activity_lock = threading.RLock()
        self.recent_activity = deque(maxlen=3)
        self.activity_secrets = ()
        self.buttons = {}
        self.fields = {}
        self.status = widgets.HTML(value="Ready. Choose your settings, then start below.")
        self.activity = widgets.HTML(layout=widgets.Layout(max_height="120px", overflow="auto"))
        self.endpoint = widgets.Text(description="SWHub URL", placeholder="Address appears after starting",
                                     layout=widgets.Layout(width="100%"))
        self.connection = widgets.HTML()
        self.output = widgets.Output(layout=widgets.Layout(
            max_height="180px", overflow="auto", border="1px solid #888"))
        self.progress = widgets.IntProgress(min=0, max=1, description="Progress")
        children = [widgets.HTML(value="<h2>SWHub · KataGo " +
                    ("Contribution" if self.contribution else "Analysis") + "</h2>")]
        if self.modal:
            children += [widgets.HTML(value="Modal supplies the T4 GPU. GPU use requires a valid "
                         "payment method. Closing this notebook does not stop the deployed app."),
                         self.button("login", "1. Login", self.login),
                         self.field("MODAL_PROFILE", widgets.Dropdown, "Workspace", options=[]),
                         self.field("MODAL_ENVIRONMENT", widgets.Text, "Environment", value="main"),
                         self.button("profiles", "Refresh workspaces", self.refresh_profiles),
                         self.button("select", "2. Select workspace", self.select_workspace)]
            self.fields["MODAL_PROFILE"].observe(self.invalidate_target, names="value")
            self.fields["MODAL_ENVIRONMENT"].observe(self.invalidate_target, names="value")
        if not self.contribution:
            children += [self.field("MAIN_MODEL_PRESET", widgets.Dropdown, "Model", options=[
                ("Medium transformer (recommended)", "transformer_medium"),
                ("Small transformer", "transformer_small"),
                ("Large transformer", "transformer_large"),
                ("Convolutional b18", "colab_fast_b18"),
                ("Convolutional b28", "latest_b28"),
                ("Convolutional b40 Zhizi", "strongest_b40_zhizi"),
                ("Custom model", "custom")]),
                self.field("ENABLE_HUMAN_MODEL", widgets.Checkbox, "HumanSL", value=True),
                widgets.HTML(value="Keep HumanSL enabled for Career, Subjects and Game Focus.")]
            custom = [self.field(key, widgets.Text, label) for key, label in (
                ("MODEL_URL", "Model URL"), ("MODEL_FILENAME", "Filename"),
                ("MODEL_SHA256", "SHA-256"))]
            advanced = widgets.Accordion(children=[widgets.VBox(custom)], selected_index=None)
            advanced.set_title(0, "Custom model settings (only for Custom model)")
            children.append(advanced)
        else:
            children.append(widgets.HTML(value="The training server chooses the model. Enter your "
                            "katagotraining.org account in SWHub. Finish games & stop in SWHub "
                            "before stopping this server. Sessions last up to four hours."))
        if self.kind == "colab_analysis":
            children += [self.field("TUNNEL_PROVIDER", widgets.Dropdown, "Connection", options=[
                ("Automatic (recommended)", "auto"), ("Cloudflare", "cloudflare"),
                ("Pinggy", "pinggy"), ("localhost.run", "localhostrun"),
                ("Local only", "none")]),
                self.field("USE_GOOGLE_DRIVE_CACHE", widgets.Checkbox, "Drive cache", value=True)]
        if self.kind == "modal_contribution":
            children += [self.field("ENABLE_CONTRIBUTOR_LOGIN", widgets.Checkbox,
                                    "Controller login", value=False),
                         self.field("CONTRIBUTOR_LOGIN_USERNAME", widgets.Text, "Username"),
                         self.field("CONTRIBUTOR_LOGIN_PASSWORD", widgets.Password, "Password"),
                         widgets.HTML(value="Optional extra server login, separate from your training account. "
                                      "Enter it again for each deployment. Keep the server URL private.")]
        children += [self.button("start", "3. Deploy" if self.modal else "Start server",
                                 self.start, "success"), self.progress, self.status, self.activity,
                     widgets.HTML(value="<b>Connect SWHub</b>. Copy the full address below into " +
                                  ("Tools → KataGo Contribution → Settings." if self.contribution else
                                   "Tools → KataGo → Settings → Remote, then select Save.")),
                     self.endpoint, self.connection,
                     self.button("logs", "Show recent logs", self.logs),
                     self.button("stop", "Stop Modal app" if self.modal else "Stop server",
                                 self.stop, "danger")]
        children += [widgets.HTML(value="<b>Full setup and troubleshooting logs</b>. "
                                  "Wait for an action to finish before clicking another. "
                                  "Use the notebook's Interrupt execution control if setup stalls."), self.output]
        self.view = widgets.VBox(children, layout=widgets.Layout(max_width="850px"))
        self.update_buttons()

    def field(self, key, widget_type, description, **kwargs):
        field = widget_type(description=description, style={"description_width": "120px"},
                            layout=widgets.Layout(width="100%"), **kwargs)
        self.fields[key] = field
        return field

    def button(self, key, description, action, style=""):
        button = widgets.Button(description=description, button_style=style,
                                layout=widgets.Layout(width="230px"))
        button.on_click(lambda _: self.perform(description, action))
        self.buttons[key] = button
        return button

    def update_buttons(self):
        for button in self.buttons.values():
            button.disabled = self.busy
        for field in self.fields.values():
            field.disabled = self.busy
        if self.modal and not self.busy:
            for key in ("start", "logs", "stop"):
                self.buttons[key].disabled = self.target is None
            self.buttons["select"].disabled = not self.fields["MODAL_PROFILE"].value

    def invalidate_target(self, change=None):
        self.target = None
        globals()["selected_modal_target"] = None
        self.endpoint.value = ""
        self.connection.value = ""
        self.status.value = "Select workspace to verify the new deployment target."
        self.update_buttons()

    def perform(self, label, action):
        if self.busy:
            return
        self.busy = True
        self.update_buttons()
        self.status.value = html.escape(label) + "…"
        self.progress.bar_style = "info"
        self.progress.max = 1
        self.progress.value = 0
        self.started_at = self.last_activity_at = time.monotonic()
        self.last_render_at = 0
        self.recent_activity.clear()
        secret = self.fields.get("CONTRIBUTOR_LOGIN_PASSWORD")
        self.activity_secrets = (secret.value,) if secret is not None and secret.value else ()
        self.activity_stop = threading.Event()
        self.render_activity()
        self.activity_thread = threading.Thread(target=self.watch_activity, daemon=True)
        self.activity_thread.start()
        with self.output:
            print("\n" + label)
            try:
                action()
            except (Exception, KeyboardInterrupt) as exc:
                self.progress.bar_style = "danger"
                self.status.value = "Action interrupted." if isinstance(exc, KeyboardInterrupt) else (
                    "Action failed. Read the details below, correct the problem and retry.")
                # Passwords are never printed in tracebacks or saved to notebook source.
                message = str(exc)
                for secret in self.activity_secrets:
                    message = message.replace(secret, "[redacted]")
                print(type(exc).__name__ + ": " + message)
            else:
                self.progress.bar_style = "success"
                self.progress.value = self.progress.max
                self.status.value = ("Server address ready. Connect SWHub below." if action == self.start
                                     else html.escape(label) + " complete.")
            finally:
                self.activity_stop.set()
                self.activity_thread.join(timeout=2)
                self.render_activity(finished=True)
                if action == self.start:
                    self.clear_controller_password()
                self.activity_secrets = ()
                self.busy = False
                self.update_buttons()

    def run_action(self, name, settings=None):
        swhub_run_action(name, settings, on_output=self.record_activity)

    def record_activity(self, message):
        with self.activity_lock:
            if not self.busy or self.activity_stop.is_set():
                return
            # Keep the preview bounded, strip terminal escapes, and escape HTML.
            message = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", message)
            for secret in self.activity_secrets:
                message = message.replace(secret, "[redacted]")
            for line in message.replace("\r", "\n").splitlines()[-3:]:
                line = " ".join(line.split())
                if not line:
                    continue
                self.last_activity_at = time.monotonic()
                elapsed = int(self.last_activity_at - self.started_at)
                self.recent_activity.append((elapsed, line[:180]))
                stage = re.match(r"\[(\d+)/(\d+)\]", line)
                if stage:
                    self.status.value = html.escape(line[:180])
                    self.progress.max = max(1, int(stage[2]))
                    self.progress.value = min(self.progress.max, max(0, int(stage[1]) - 1))
            if time.monotonic() - self.last_render_at >= 0.2:
                self.render_activity()

    def render_activity(self, finished=False):
        with self.activity_lock:
            now = time.monotonic()
            self.last_render_at = now
            elapsed = int(now - self.started_at)
            quiet = int(now - self.last_activity_at)
            timing = f"Elapsed {elapsed // 60}:{elapsed % 60:02d}"
            if not finished:
                timing += (f" · No new log for {quiet}s. Waiting for this step to finish."
                           if quiet >= 10 else " · Working")
            lines = [f"<div><small>{timing}</small></div>"]
            for stamp, line in self.recent_activity:
                lines.append(f"<div>{stamp // 60}:{stamp % 60:02d} · {html.escape(line)}</div>")
            self.activity.value = '<div style="overflow-wrap:anywhere">' + "".join(lines) + "</div>"

    def watch_activity(self):
        while not self.activity_stop.wait(1):
            self.render_activity()

    def refresh_profiles(self):
        self.invalidate_target()
        # Reject token overrides before interpreting CLI output as profiles.
        import os
        if os.environ.get("MODAL_TOKEN_ID") or os.environ.get("MODAL_TOKEN_SECRET"):
            raise RuntimeError("Token environment variables override workspace profiles. "
                               "Remove them or use a clean runtime, then Login again.")
        result = subprocess.run(["modal", "profile", "list", "--json"],
                                check=True, text=True, capture_output=True, timeout=60)
        profiles = json.loads(result.stdout)
        names = [row["name"] for row in profiles]
        self.fields["MODAL_PROFILE"].options = names
        print("Choose a workspace and select it." if names else "No saved workspaces. Login first.")

    def login(self):
        self.invalidate_target()
        self.run_action("login")
        self.refresh_profiles()

    def select_workspace(self):
        self.invalidate_target()
        self.run_action("workspace", {key: self.fields[key].value for key in (
            "MODAL_PROFILE", "MODAL_ENVIRONMENT")})
        self.target = globals()["selected_modal_target"]

    def settings(self):
        values = {key: field.value for key, field in self.fields.items()}
        if not self.contribution:
            values["HUMAN_MODEL_PRESET"] = (
                "official_human_sl_v0" if values["ENABLE_HUMAN_MODEL"] else "disabled")
            if values["MAIN_MODEL_PRESET"] == "custom":
                from urllib.parse import urlparse
                import re
                if urlparse(values["MODEL_URL"]).scheme != "https":
                    raise ValueError("Enter an HTTPS URL for the custom model.")
                filename = values["MODEL_FILENAME"]
                if not filename or filename in (".", "..") or "/" in filename or "\\" in filename:
                    raise ValueError("Enter a model filename without folders.")
                if re.fullmatch(r"[0-9a-fA-F]{64}", values["MODEL_SHA256"]) is None:
                    raise ValueError("Enter the custom model's 64-character SHA-256.")
            else:
                for key in ("MODEL_URL", "MODEL_FILENAME", "MODEL_SHA256"):
                    values.pop(key)
        if self.modal:
            values["KEEP_STREAMING_AFTER_READY"] = False
        return values

    def start(self):
        if not self.modal:
            for name in (("service", "tunnel") if self.contribution else ("ws_proc", "tunnel_proc")):
                process = globals().get(name)
                if process is not None and process.poll() is None:
                    raise RuntimeError("Server is already running. Stop server before starting again.")
        self.endpoint.value = ""
        self.connection.value = ""
        for key in ("final_wss_url", "final_status_url", "websocket_url", "base", "ready"):
            globals().pop(key, None)
        settings = self.settings()
        if self.modal and (self.target is None or self.target != globals().get("selected_modal_target")):
            self.invalidate_target()
            raise RuntimeError("Select workspace again before deploying.")
        if self.kind == "modal_contribution":
            if settings["ENABLE_CONTRIBUTOR_LOGIN"] and not all(settings[key].strip() for key in (
                    "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")):
                raise ValueError("Enter both controller login fields or disable Controller login.")
            globals().update({key: settings[key] for key in (
                "ENABLE_CONTRIBUTOR_LOGIN", "CONTRIBUTOR_LOGIN_USERNAME", "CONTRIBUTOR_LOGIN_PASSWORD")})
        steps = (["write_app", "deploy"] if self.modal else
                 ["prepare", "drive", "write_app", "start"] if self.contribution else ["start"])
        self.progress.max = len(steps)
        self.progress.value = 0
        try:
            for index, step in enumerate(steps):
                stage = {"prepare": "Checking GPU and installing dependencies",
                         "drive": "Connecting Google Drive and restoring cached files",
                         "write_app": "Preparing the server",
                         "deploy": "Deploying to Modal, waiting for startup",
                         "start": "Preparing KataGo and opening the connection"}[step]
                self.status.value = f"Step {index + 1}/{len(steps)}: {stage}…"
                self.record_activity(stage)
                self.run_action(step, settings)
                if self.progress.max == len(steps):
                    self.progress.value = index + 1
            if self.kind == "colab_analysis":
                url = globals().get("final_wss_url", "")
                status_url = globals().get("final_status_url", "")
            elif self.kind == "modal_contribution":
                url = globals().get("websocket_url", "")
                status_url = ""
            else:
                base = globals().get("base", "")
                url = base.replace("https://", "wss://").replace("http://", "ws://") + (
                    "/contribute" if self.contribution else "/katago") if base else ""
                status_url = base + ("/" if self.contribution else "/status") if base else ""
            if not url:
                raise RuntimeError("No connection URL found. Check the deployment logs below.")
            self.endpoint.value = url
            if status_url:
                self.connection.value = '<a target="_blank" rel="noopener" href="' + html.escape(
                    status_url, quote=True) + '">Open server status</a>'
            if self.kind == "modal_analysis" and not globals().get("ready"):
                raise RuntimeError("Deployment finished, but KataGo readiness was not confirmed. "
                                   "Check recent logs or open server status before connecting.")
            print("Connection address ready. Keep it private.")
        finally:
            self.clear_controller_password()

    def clear_controller_password(self):
        if self.kind == "modal_contribution":
            self.fields["CONTRIBUTOR_LOGIN_PASSWORD"].value = ""
            # The existing deployment cell keeps its subprocess environment in globals.
            for key in ("CONTRIBUTOR_LOGIN_PASSWORD", "password"):
                globals().pop(key, None)
            environment = globals().get("env", {})
            environment.pop("SWHUB_CONTRIBUTOR_PASSWORD", None)
            environment.pop("SWHUB_CONTRIBUTOR_USERNAME", None)

    def logs(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before viewing logs.")
            app = "Katago-Contribute" if self.contribution else "KataGo-Analyse"
            # Recent logs return promptly, the deploy action already streams startup.
            subprocess.run(modal_command("app", "logs", app), check=True, timeout=30)
        else:
            paths = (["/content/colab_katago_contribute.log", "/content/colab_cloudflared.log"]
                     if self.contribution else ["/content/katago_ws_server.log", "/content/cloudflared.log"])
            for filename in paths:
                path = Path(filename)
                if path.exists():
                    print(path.name + "\n" + path.read_text(errors="replace")[-12000:])

    def stop(self):
        if self.modal:
            if self.target is None or self.target != globals().get("selected_modal_target"):
                self.invalidate_target()
                raise RuntimeError("Select workspace again before stopping.")
        self.run_action("stop")
        self.endpoint.value = ""
        self.connection.value = ""


def show_swhub_panel(kind):
    previous = globals().get("swhub_panel")
    if previous is not None:
        previous.activity_stop.set()
        for button in previous.buttons.values():
            button.disabled = True
        previous.view.close()
    panel = SWHubKataGoPanel(kind)
    globals()["swhub_panel"] = panel
    display(panel.view)
    return panel


# Load the bundled actions without executing setup or starting a server.
import base64 as _swhub_base64
import zlib as _swhub_zlib
_swhub_action_bundle = 'eNrtfWt327iS4F9hq3dOU4lEW+7EnWhWd67jKImnE9vHdrpnx/LwUBQk8Zoiefmw4+vr/75VBYAESJCSM927+2HTndgk8SgU6o0C8NgL41UQ9cZW70frr3mQh8y6YKsgy1lq0SfLTosos+IofLCWabyx8jWzEi9iYX8WBZskTnMrzgZWygZWVsyTNPZZls2iWfSj9TleQQsrC1rxVh78G0R3XhgsvJxl0E6QWVGcs3kc3/6UWUnK7oK4yKwFS8L4YcOi3Mq9dMVyZxZlLGR+zhbuJl54ocvfWxPrNI4Y9sWiO3iKMwd+CdI4cvw4ebD79OF61js9c4/PPp9dzHo3UGzWG8168tPV9OKLfL0oNnP8Mov40CfKiJzzOGGRPYss+AP1CJBZbwDV8viWRfzXiN1DYwNeKssXcZFP1DZOzqfVR5am6sfLq/dnX6/E55x9yydXacHEMwA7gb/wBINaxjA5QcQQsQSpw7sa87JJGkS5jQUGUG8xgSH1+vzTxsv9NYwrZU7GvNRf2+mst87zJBvv7fEhzQB3mz0a03AZxvd710fD//SG/9gfvnWHNy9xnNi0aDBY8jZF11X30OksOjufnlpXn04urc8np79ap2e/j+HtrGe95LWcVRoXib3fR1pairHce0Fu960fJta+aDX1ggzosojyYMOmaRqn0PwXhFbQ6NILQrZwrOMw8G+R7ODdnAGamMUpB6nQs+7j9DZLPJ85hBAYSa98ZWaB8vOubKCzwJez90ef3fOLsw8nn6dEYjB2C7pJvNTbWI/5Q8LGs16WA8pWs96TrDE9/e3k4uz0y/T0imptgHm21USGOw5hUmGgJS+J0cclOrw8Z5tE4GMRLJcsRUZTMLOV1xZsaflr5t+KAn7KFtBG4IWZ3R+XZKEwI9THCaOhXZ39Oj11T97DDFhAxp2lLqfHF9MrKKlQl4kWyo/4p94R9tJs1IrvgP2CBQiiao5h5pZASJmDE1Vr9IJtoApMewzdC5BJRt15aeDNoRZ2VMBHz/JhGiIr5TAOkFTgicEzJ01Hbb2PKOVIFeiMNxsvWtgvAO8F9pDJ4bcivcS5eepAzuLcbUFiDx6tS2pAQUlW+EjMyyIEwhc0xAU0kNDAugvYPdISsCGNP8vjJIEXTilxBE4HGs4mZkgFt7O8SKOajB0ORUP4WLZZ4YiXgT7wu9LVDUdvB+oUCQwzZOvdKp3OeiHIBJTufCJIOkN9UQTGpHG7g6yZ2ETkQZQUSNu/S7TuyUqRt2GVPMEOLG8OdDaG7vqyCVJVCu4aMqIqCDQAKlUiCPvGR6W6JlF/88KinP5phAJvHudrVVLqkAJdwl+1PY26SaX7LAz59KNEumDeYkhSk5BmAdcFywCYRZkFalYdopRV0YNiDDjY3PRbAiI+yK1l6AHFQeXYumUsQbmbsb8XWFswEAprDpFkgyUh2QN5eFfyOoDlRStg+QYhmDS9mQpnPS9JVBJpJcaBwshSu/fbJa5tYh4iOdSwS1Kxl6Juc8rG1qP47cn6pzVVaMB6VNp7ormSSvs94XvAGRrnBTm6QmkQ6XabdR+EIUk9el2qG0XBpkHOXEBQi4LFzxZ87lSw//IvVI4mjOPo1su9VQzCMgLSnxc5c5IHtCzwv1/h28fYuhs5ozeg4zJRYjHMgewjFFdlNWAQNKYuf/9UzJFeyKhwkHQ/soilHiIWscDJEB7mDxY271JB91i2g0MOkodo7lhXsYWkjeUtbwljnEUwMuq25I8BomzMhyLa7h5XdIVclniAM44YnC6c9AwNN+vrxWcrCUHf71UVYa7ymI/sJzBGBFJUiKG9FSp8jjXsRRgyXvYQ+UFcPq+9bB0G8+p54/nlw9+yOFJMoPLXlJW/ZmvoL6weFRtJvEKjhPhKvijSEHp0UuRpYCn5+h+BKEYkkng5wmWJj+fwKL6I6mAsAWlWTdKzMlDO2qQhjs7P3dOjL9xK+5VmYVgiiyF6uNj9eP6Vily9wneX08vLk7NT9+rkyxSMd/fT2deLS/j+Cpv89ejq6OOZezH9PD265A0TTToHWFV8xakDTueCobLFV0G+LuZkiofBap3fAV/F6R6fxb0U+MzLWLa3iO/BV/EWe6LhPWlazHqckobiw9AvFt7owHmDv0TRW+eNsz8EO774Nvx2+MoBvGLNfgnW5aejg9eHBPOBPx+9/WXpv537B/uv95cHb0ejA28+fz1/87O/mHuLw5/f/nzoz18d7r9eHLLR27e//HLgsYX/9s3+ob9/wEnr+Oj4E+IAJwnkzJ7vgRQkKSF6fH9yAZ95sb0Sfvdu5I7euAcuh99943L4FQweXRx/OvkN21ZaqlqQYxMf352ctpVUSk3/4+ri6Phq+t5cPvt7ASyxzIZpHIO8p3dHSQIWFLZxfHZ6dXHy7uvV1H0HE98Yma+R1aezL1AE2m+UW8cbKvH57GPjG8pnjliUnRNOyQ7AYEtCBszexSFYR+XX3+jRQQZxUZ3bJZGoEA1QNYPgc4OluwmyDASXNHUCYaM6QeaGse+Fpb0PbmYSg8BxvSJfuxmDJvKy30t65P0uAj9XDPbHupkN8urrO7fE4NmF+/VyeoHjmYECqXsLWuXtDQwsLujUGv3BdhDOjy4vfz+7eP/dIFQNbAPhif+KNl+Yse/C7uMTuaQg2yvBwoue4EteMiUFnD4oA5j1YFyLwNtDVhujrHBGXFoMhSszLObwS3Fw4OyD9FOg9hYLN3kA3yiazHo/O6NR+VV4AYAz6IwGYV/fyJdekrtBlOVeGGpw+N7QZ2kOhqKPoSKtK/gKgrz2CmT9Kt4ko+ZreDtcsLvmB5AKr2pvk1/g5RBdndqHIiIZor27R3+lPsokSKoB9Zagrr0kuIZ/o4WXLm4mk31ndHiA8h/poLgL/DiN8O3rV84+f3nP5lns37I8m0xGvzivSEQqTqKbrT2QzDbqvjFJ0741/AuYaakglkUA5iySh9DajqggYLwHxUKK0+HxrF46R1fcA+MtLlJf9RLRMvLXRXSLZh+YX6kdeps50IYoCtrZW9ij/YNX1gsLf/QH1pyCTWOdwDlITpFg5M+mJvuarycKrNk3/putDljqOBvmfYwDHQiLng8fjONvCTef+UipDCFF8XpFJBEMAHQlNreLILX5Q8YNcWgFGMKNS79OuNSiIn0Ff5GsQfEO5hUghR9uFvyDWX8p41WiqpwqXrxvTSYNUHU8cWxU70Q/RQSK+lZOIJpKceqlDzDFogDOqZsVy2XwzZaw0ZP1EijKyTdJ6Y6XtWWrQsTXBi7sLuhCN8ScC/5TYVcooDDGGkiCpdmkIdi/ZiwdHq0A4ShFSUgW8yG3Z4ZkTVfmFhg6I+KHeiNHvs8S0cCLvRdaiSedF0HcKNglqq8NBR6JBcQzUBWIOAzYjg72iSFSliVxlLHaLFFbFR4FG91LNoIWkjIUq/7hVjBFptGGjed/s2UPA1FLwM6+4TDB2cUfYKgrje04f9SfyZ+Vf65NikuTafqnIThP4LZPfm7/LrB3uN9apEAa8JAGJt8x/aKVM+MHYHm7xE3fUEAnU/xzU3vWvHIlPCe4GQMpFfZLeQAysnpbkwiTiSYSDFE38OHfC/GG8YRF4TMeYdkkOfjB3IsH0J9KBvb8vAA/kcsP4M9SyJSDLyHWi/7QJX2eQVZbA7DqmAinWbGxoD2+9oBKhUYEA9MAfKpFRXWwUnCSPZ9JWaqoBxGbcDkHCVjKuOCYgmqggtMy+HK/4IrD+icpCBHZlusunIrH6D3DFxAFGKJBZZIXSciuAwx5YWsykMayIsz1lSKd26oQZfUOYJjA34E6Afp6z9b1ox3WkCqi/uCBMan2JnhV/NQlp9DJfGQOf/LjBRvIVxyuagpYlBUpE6ELm5CF+BUYqlyn56hd5LbKYyvZTSHZyi6oPOmBpfuE5TN3Z/s1hSCiCc5/BskH+GnrlUma4xpZcFdXAeItgJWnQMRo7lXDVHrRG+zmrZ0GbRYhIrYjoAJrakFNoVeHq66PVZuVHFH68dfgHtj78S+vXxNjEd3QhHOeQilT4zJkKLtqoo+W6x3ofVBWs95NNSJshsQg2U16QARIC5dUkpSh3QL1UYECuA0FKgiy6k4CqaNM89lNyONjEnPmto6tpigNIx0OwekmD2soWqsWfVUWr2iiqVFUBP2g64mdpCzXhmLmwe8n304ODoN7fE20XG/Vh11zQPkjRusz0L6ggig6k8EPXGAGQBKWEtXGkQirMwtcwQV0AA0zv8hxBczK49jhTdnGOeGcT/GLIktF3GROS5tqFKZfo0iFXPX2msWeQbZaU90EjPODin4H+kV6NNKwacVNC1SLzAjD1Gm2oc4M2iAUpTiP49C+w+WdsQWGJmj9mpcompn1cpBA0AsMlYrjYiEKJQvDDxb6sPCTB5pE038vYjAhdmkcI8POotgkmY14pxr9gVQXXuYHAddMqjrHMEcQuSmuythyUqNlsBpbGN0QjHTLHrgrKHxetvRAM5HWFq82QRRsio32yvumviKgg3JJTHcYODIm+N3m/VO4B/odyN76urVuXz0kfGYHysIaKREo0SAFZe0NJPgjNPxkbYoMF78sz4qKzZylSFAUzoYGSorkkP1POUCkS/7qL+UAn9PXnOX3jIGSEO09kaR+FE1V+kLMKXWlTBdY7C5avczWZqmcoBplSLQiPdTROut5YPtDh30njO9Zqqxr82qCtx5lwUFJvoOKUp+eh2loCBw/skRwLYi30TpkXNy9mhK/Eee5v07/F4b7H2XQHUNnC3YX+CyPC2xqoH6BN8tkdNh8G63vfeXtIshQnoJkkTZOGMeJUgAjwyArPDCnlLdB5IfFgkEVEGTrQK0Qxiu9LLxYeRuWeRmt31QfYOJTD/NDyFxXgm/0KQOiCHMvAqsxo/rq5wBxDKMBY/Tbg/oBZGgoR8JLzb2MqWG8WQ8oPgJPOAUhnK8xsJShzsGaSiFcHQQ/4KENwsTLsvs4XaivEBjoLvAxeLq1ALqoGBuvF1jHWV5/h0tIyjtwUeAFQA0uDigdnzEYi4oHPjp92LmX3ULFpYfLO8p7Axz3Xhph8SKCjwsg40z7CLiIo1UMeMGJCSKxTN1doOziSY0wekvmkr3gYqZXZqfevSbpS5eqzCa4d9GBEYwNj8hMStoZ5nZhpEWU64OsGh3u7+938eqsd7RYBKgPvVCogJJtR4cDqI1r92jTALUT87J7Epm8TYJc8f4Atusb/klNn5MgOVkSBjkfrmozUmZFAmbOhKpUmRaKDYdyqSxHSTD8dwwHpHmGrgaM5sdmXJS6c8CWZNHCVpLqKoMVzIUIhY98o+Tv0a+YvvdftsjQu1Ey9W5e9Gez7MUEZaOEpwl1PXXPPBENuxPE6AnP5MTIf22OCLO2DDWgIgB4ue4Hlx+/PjUWQDTA1MzAUakLcLKM4ve7wH/UO3lCswd4FkxoWuLn+QCoCMECRkQBzYmEsQWLOsBvm1GhSuhzxWnzIggXLsebpj9rWpMLDhckR1N1SrHyNQ3l6lK/r5MprXovXK0VuRhuV28rXlVw1qjrZCBzN2QTi9VqFR0i4ahZK2J5GPtawWYhKfQsnmEiMg47q0iB3qzS7xYuJdYUq4uTtFyCx6yKUpyUoBkmQJHVLfgXMBrqKgpJ1NVc27LbErG8ePfYylpIwWXv4HxRXD8As0LkZ4kEvW8PLuo3I3z49VMss5oMg6PqXeihEu9QwR6Bgv26DVnUXifKtPbOWxDI7Yw2niHDQkYpv1BRaJF1MBEKprLJcYNBtO5K5qre6iKuJpQabXQzWZPRlJp1RtM5RynYzWyt1doZrs50LcTZjvqSEyUHltxHxosrCIxnkKgaWp9bKnzGLZ2PUOUkwuDeKdFcnxsmVAStICf/lpe99B0iTOkrumTgYhTB4JNWfumAW8aXimX8UVjG1puBNYL/D95oAVZsm1uwjca1Ji+ozJfSysW29qk5GR4Ha9AF69Hl5mNHa1jygiUfpJlpvRpYB9DUYamf0HZ1FePVJet1p9HzyudV3akwfCl8jlAfoLWn4SCKXGHm79QFeAenp5cksq94Nejui/AOaCQ4GK0HdK1c7ot1IAZLvadCV/FXctgQxQh1FQsl4wXhVBbNlqoGAQdQhkUUdfqkLVQtFbGslJfvmqUVEViWlu+apU0UiDVLOn6qp0DExBzZUfbv4ADiWjL3pLVWhUd5Jj1KgoViSwrH/VQv9dPA4mGdfr1TM2vyjRFlsZdWo4u2mjhdoisd8HYZoKKzEisNSNHTPAdPU+PBXQA1VmyFs87lcsq4eKgDJeIC51Vc4HMcJ7tAZa7ZCpYmLhCmmqRpkJ9ZBGDNNsnSJI00uoJevpJ3+yt4t7uRRb1W65jaRQiCWcmjxthqIgJLK7LFWByKfTgfHRKt1SNkA+snpchP/bYGTj/9frylASxiaACjQ++93HsfpCq1a4mGzVogEGoVRN6hUvSmtIUqq1Hb+UW2G2bWtktcKnTOQycWVw+Hr1///LpfC75muLqGbtTWxIWlYqZq0rKEsTbaesVzDvFjBX+z/E3bCo5uANfsn66RtIxGBaxhNTeH16JAOls7N6kWze5uae3G6Pi2mcbPn8YO21ABVbGqn1qyRHgRwNo5DqsmSTolil4TaZTWaPtbyUEfbTOIpvVCH47pDboa/dqiDK38OH8D7cVXTbI+ZXLRax4++KuXJM6yiGitT+CWViQn9K+AdpUUkzJtXeY/wLtXckWEbeL0YfJm9PZAz3+YmBPbX1iH+/RPuYnVS/MicWWtA+0rKjKxBo0pYSNZyfdCUkPufRDBz0lZgecrY/JYmR8+Fi9ldhdPfs0m16ac2Bu+neavPNsVuvaLFFccbYSENmEBFFUBL1sFuDMFXTwMx6xWZeiPVl1E/qbcP/ABHo/OTwbW72x+SSmayq/vgwz6i5ifS6vxns1dnpstKtqlQflX8U0QQ+Xn0MYLXF9CzOZFbe2fU0cjsy6+RTTVckf4J9qXNLZkSnjju1xxHdeWNhsFgZCwVJ2WlNzZNM5jPw55bl52vy7mw3pyeaAsNDRr/iZXX8fWqFGozIrl7e9pCev1wrg5iZc75lMiUvbWTAT22vfC9PRU7Np0lVDYNRAa01d9tMtK44pa1Gn1cL+zVZZyPMpxVEPMIrlnLPflOGpGEP+p5zLVPSi3ShZuK5hS3qaLdp6Mmsve0L5Sw+cUDktQQrNF6SPvKzFQzNnO+RcezrympF38Ddt5fFKcd2Bj3EpwC+9lf5/hUR0+5k2iNUwJP8ROuEinBGDQ9Gu+h6azALf6uVyENtcCqrI4ZWAag21r42KAaeWyQpRI+MKCtcS3cik2iJrdN/WPXNgcm9V22VOZicVT4KgOWlLXF9P3tPh/U1JgTVbwmupIFQmDagpMcRChtJmcZ1a/eIHrivXs6QrXD5TWhyuePayFbFa1Ias/1bKlqE/iwXK+DWOuMwKVRSzbZvwo6QUCrgG0j/vjwUPJJjYyMwXzxlVUsGa9mPDCNyzKXdjipzGnXDcCiQ3FVmsKnQkOzfIFps9nppqmpG+F43ldhyCyJSiVJdB3GCXm2fXRcUzqjSxw76Pdr1O9j+k9XDe6tFvRpc3rRVJjBAPwP0o5ChXuQAgEecblXhxa3Jey5jjVwJyAAFB1aAfZfhD1LS/HvL/cqTc4Pftgyb1t0J6VgZB++RKBYN7GEpDJNRncuMt3WFpcCgFy6g1yOZVR7HsBQ3CsX2m3MM0JZm5bmJET8p2ZyyDiJ0pgv8DMPgbL83qTKcPTGKDJOy8IedrTOo2L1RoMIGuVAocui5CEFsorp0EqtNoIpC3XBY/CkI8M8FQkAwv7zOlQh75ZZii0JgOwPNBfJzj5tUW+6PRBhGC38ASSXpC7ZJPZbfSgJ9DotChlex4LN9vmZp3DW67Rbz3/vZlDU66HwgD4RifwHvj+NgGshaEBQCNudv7m/5A+mTwAvutZNDKwlmGRrev5zt3Dq4ZI8pTcaJQ6os2J+GloryPJXxe5/CiR5qQkLA1iUKouH3DrvNyvMca2jWqqPN8OWaVPZhYCL9kjbvP320rrtFOXP8BzQbZ2UWu4tDGNcyzoiFVN7MMb8qTUvC8S41VytmJm/NM0hObpNwZxjMJ1Lu0bK1stZ7PspT2bXb6czRx47Nv/NoZX//wffaPvKcE0hRvwzIIJdH7y8fTsYnrcNLT7DWnBQd6mPQyLNwApmX1ywURf8K51pMyCarHxlI8bru55GVw65M7G0y7cj1DIzBABEO0a69LrpNRw1+msV+TL4Rs6RAGd8mxCgUY0gkzMbBYiZ5e08NSOuhKuLGebgYoKk+HnGk9QupjOZtf29X/NZjeUeHFDaRdi7M1J1ZraOrfPA7C0F91aHkUjc0UDxtC90tE1Ip5MSTopS5qVjT02Ccgkt2630lu7b9Rlag09XebdS7MCbEJ2H4CTl8ozvN6ZSIOFnX39/p19/W7qq4G2YBUh12AVDQTFBnh50WbBd02+SSkQc/ENGq0KIYoj2r1d8+LqRnvG0nwXYwPtntY1ZK58UF4Y8EupQpOmuQotkpQgg9FMONgbaYM236lpVNcy1Sida8HIgC7FC8xVnDLt7J16E+0Wc5neZjAeeJ/Cx+woWG40+5a3DK2ZFFY3LjXafmwn7XaLhrQz0BiATF4WGdF2C8hK90FGu58j8FKx+kAkM9EJKvAs4l1IbSeLdrhMgQTK3Bc8Jevf9JGnaM87Oa3drWlW2kpkoGC9yQ6V25GuqBpC1b9fnp2+J6pq0zu6ZWfAIug91fzo0ak7OLEiFa5TUpX81DS520HpsMIGJc+Yq7cgLpCm3Q6OCEWqYULcYKFJOLTq6am9GicR5GaFVpwkTuyyQey4m2pFI7sBWqGLgk6PL17w6hj7UGB/+l5ilG23gFwLvL2cWKPtTkkrKBWltS4eqZ1SwsBEB6GjmvSB7C3YnPW+SnrHJq1HvQej56av42BivzUr9vfnv1iPHIPX+zdPfF9Lkw7l5paG0u06GKSD4JWjOnbxDXk4Y4KnN9JGSsVdFCK2S7ODrmrV7LsrbWjlD1ba0OL/V9rPV9ok00U4Wd/3olFYGWOtznRpMjNhpI1buGKJfK9YrXOhsmi9hZ8ii7nzxnqUMMfTmsQM4aY/3tMuVXl2PK5m8c10B/v7Fi5ZbakrWJRYZquvV+M0NSKtMhtAvSOzUbRU7Bp0+dpI1sp1ejGimGox5Xp8cNPwuRAneq0dA8PuwMLUcp46qOt5OjZXb1Q50qIen8G1CoQPZ0G0aAABSzg+2nOhbYwxy85X4KIA5b6o98/H4JbUlsnomnFFIo8TV0gUm+KvYwsX6Lf6MdoK0UCsCD0nRq8HvzqsANOc1NencIS7xEYEHdBARaY2wt2pSPTFCe6Vb1irHmtbG+tutAp9c/1JAAp+5K/NoRb7XYqnRp8H5T5Isf4KE3/BMpbzHZGGATZN4V2CyfjgLo1bPNT4Nj9PukWblwzyujZUfvLDbjGmcqWUN9Zm+Lf7WxLUHLdZR5jq1R4wFiu+n+P4tki6fIw2dbotkN1Abw2R7ULl+SjZ7obKzm+DMLT7nf7Xzpjpwk5z/YwPWxVZTYg7re1Zb83CMDaa2DL7YfKcpAm1okibmIwGrasjk6U8Lf2xTOB40s5ARXn9QDpBLOk96vkgT04Dhnq4yrs3aiQioGpNN2U+wzVxCv8qpNRYPzATRXUslRKWgK7NbLmjP66Z6MKwqYyEWe8Swyll13KPAt8rhK07RvFbX83m62sIx+TVq30T55goUugIJa5SnlZFoZU/bDgeDUVE/P/kEckDuMrtPHr+Dx39P9oysm4LuEXYV0j4GmVFglldoB15TpCvJQMJmCxxJIRjbPJPw1F5TlOZ5KZsHfgzzsDc1r2yF+HPOP+y1r2a2UfWtEou8ivmsTbIFIiLMjrbEUjE1VKmTH79c2mPi2LZq7jBIshErlaRUr4EWGNJyHIWPvwRtDfaH41258927BkQYyLRFhRhCFeb3Nb9k60erCIItZaEOOx0JXf0I01E/8wBtW3q/L8zIKFDWsaAZ3ljFgaezeOK0zfbrabGbA86qOVZAWN0gL4TFonuQYfk2hmW/i6BPINaVU4v1C+laXXOunXFqL+7Vd/QqfwIKKFKMfTh4XYLuWlUoblWE0s05MsUdU6PNbn9HdbHhyBFq0Ok7VTWR6ZaJX+m/VHPy3z23gw64YLDeV0h6KaV9Qff1cyWPc6D/0cF0k57Rtpdp57Ioaf0N5VszlNM7MSwl3pmlnOA4SvSpwM8vmGBmZ8PuimH+zBycftIhnfyyNtF+ALKnAGrzoqD/YPDBtFxV6xjQU07+NBwdocEAmRZWts/Kv8Yzoo34FDbumX4LnZqte9RkSE/AIQuEnhAxDVANPBzWWGnwxsbKMDkcxFG1w67MFB+DX8LlvlpkOQYRiqPAZVbwOV1EQAU5kUktsk/xmPZpFutXKjhms+PpfOSobjjL1fGEjB6+LxHhykPzIsFE8MmjL4RJ2pmlj66vuGwYbC5lws6Z1jFyqx3z++XqadLdR9DzD+IpGJlitoBLc/ZOyQpWz8KUxxvaqZvlEiCP4wbtTo2cvACQ0kgLYf+KnCaexjiWVND9fSrehMGHux3s5IMJddFg7g5odqf4eLpiAbqfGE6FbaCCeZzYtrsoR//qpTHGMqzKuC5sbtXCINNkE9e74tz3sWPbkrfEhIjhd+y7rxl4bg68bQephJulJBqoC3klT5q5Msx2s1bwlvKio6Z0GskgCVtLRnLRJ1dlWidd9dKjVxgAwEbSdilWZqYG9UCoP2OrOLdsodF8K+lt7YgocHkwEz+geWal8DMNPMoGh1oA38adBnZ7v2aVXz44eTi8so9Pvty/nl6NX0/2NX30hANWncRdyyAE4wtK26dK2/l8FpX3ExNzQH/t8YFd4q7cGh4uqL9rJR4ee5qlZ/fGbD9viQqzXIU6f51p4xvmdk1aLs1v67mPtWG2R6UfUZGTjWQ7Zk1zRHW47iDZznh7WOXm48m9amte5xGtJW1J3RII52M+ax1TlFn2zJnc+nVSCtigbgnl4qFNWtw5vnOUCxjJhhKLNZGJ1dk/08u4z6DunhP6tj52fBtgxflnbaskp1wwheNWzCyfflcFaZdS+Tfjd/n4bjeuHFJ/JnTIu9N3Ynrt+TTldYR39IWZGu67Zdfe1nulOM3N4rknDS3tiXZ0d3WYE/hVZm9Pyhrrk4nSuLBLiJfQZoaYcJGhvKbiBe0ePi1Zhv5J9+hO3fMBfyDFMBSX8rSIh/l1tVH8dsP6VOXQjDtP3D5Af76ijiZMiZj35w19Uy3IMajSo1wPjMNVcGRYAjcc4qXZqJPTQN7VEbZvcYtrBPDeQ/1LdLiIlW+qoj3IzMMu8jSDDcp67SJ/Zq3Hu6eS9GVLtMejMUGFxgQsLAgQrSk42vpUA+esd+W69O1bVHP5tltM+f3DGCpR93Fqfm03/Ppvw93NZkd/XCWMs8ebScOHzo3i/93t2K2iKzyVGJNt1KUVh6435YMqB0E8ayMwGC5rbI5Z1CttKP/QoqjCqR15+ypkayttxu1T9H4eRtcxf4tcXQIvwuZ3/RrvghZ3ALcdQtyeccuYmmgXaPbfh0u3dopLrGWRsusJ8/SX4XxnO5B1+4Fr11TcRwG/m3zavj6bfDy0m8W3el5AnjjmbjF/HrWOz0DJ/rz2YXcRDdCc4J/uppefJGvF8UG7xJE6Esgha8xPT1693mqJRh8Pvt4clod/TZuHNpbb6NRuZEi0X6I7/a26vkOf8Shvh/QXkNyOOZnLISsXGgMWIhnHKzwsh/zrPD1UsRxV24IbVHUFm/b6lQDxDrVIuss4vFpNcDt+PcLu083vHTccd0rL69XWqjf4tNBnvziciuPMfKPqxSEK5G3jucwaYjhF8pzehWcp13hdc5v9avuTlXsfV5ebOSwFWjLoFvnxV1bLu2qryPADEzgLz/ZinfNDxQTh9ioh9nzz2IH5Viep4wqjHak44r5RCFHtbXameUwFaIxHvxTrygyTsIX5SpzRK7QkgLFRRq2w8sBGFdn8+MhyqSJHFChC7xcK60uxsZN1tnNy3+bzRx+e/cM71vD2SDQ1euecIcHP43ZSTkvz3rOoH/zrz9p1gEqEyoqZCICWxP2+EpiiMr2hWBVyuIyQ3X+c3Y9HN1U3SpHGcswpygrq5XH+1RDFRfCZuIoZDr4ZU+/PFq1OGez6FxcEA9qS7393VLNl9KErVKQSICMSxB5gxqg+n3ESo/HINuJ0yTQjsPnBWeF7qX35jHGJX08rkUUQy6FYQ3I6vEWC/Xaes6VqC3RtjHrSvyyi6ZUNeSP9Rv66mxNZ6UNyqu1B+Zr4Pn1X0sQv/E9pUVUt0ZCv7t2gQNo7aLeaO/pfwOnvwxR'
SWHUB_ACTIONS.update(json.loads(_swhub_zlib.decompress(
    _swhub_base64.b64decode(_swhub_action_bundle))))
swhub_panel = show_swhub_panel('modal_contribution')


## Before you start: payment and usage

**Modal requires a valid payment method to use a GPU**, including when usage is covered by free credits. Add one in Modal's billing settings before starting KataGo. A new workspace may initially show only $1 of its advertised $30 monthly credits. Check the credits actually unlocked in your dashboard.

Modal charges GPU, CPU and memory use against the workspace's credits and budget. A budget can stop the service even when credits remain. Running analysis and contribution together can use two GPUs.

Outbound data also counts toward the workspace allowance. Starter currently includes 1 TiB per billing cycle, then charges $0.04/GiB. Cached files remain in Modal storage when you stop the app.

[GPU requirements](https://modal.com/docs/guide/gpu) · [Pricing](https://modal.com/pricing) · [Budgets](https://modal.com/docs/guide/budgets)


## Advanced implementation (optional)

These code blocks are collapsed by default. The top Open control panel block already includes everything the panel needs, so you do not need to run the blocks below. Expand them to inspect the implementation.


In [ ]:
%%swhub_action login
# @title Register login (runs only from the panel)
import os, re, subprocess

# Logging in again invalidates this notebook's previous deployment target.
selected_modal_target = None

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"

login = subprocess.Popen(
    ["modal", "token", "new"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
for line in login.stdout:
    print(line, end="")
    match = re.search(r"https://modal\.com/token-flow/[A-Za-z0-9_-]+", line)
    if match:
        print("\nOPEN THIS LINK NOW:\n" + match.group(0))
if login.wait() != 0:
    raise RuntimeError("Modal login failed. Click Login before selecting a workspace.")


In [ ]:
%%swhub_action workspace
# @title Register workspace (runs only from the panel)
import os, subprocess

MODAL_PROFILE = ""  # @param {type:"string"}
MODAL_ENVIRONMENT = "main"  # @param {type:"string"}

# Clear a previous selection before attempting a different workspace.
selected_modal_target = None

def check_modal_credentials():
    if os.environ.get("MODAL_TOKEN_ID") or os.environ.get("MODAL_TOKEN_SECRET"):
        raise RuntimeError(
            "MODAL_TOKEN_ID or MODAL_TOKEN_SECRET overrides workspace profiles. "
            "Remove those environment variables or use a clean runtime, then rerun Login."
        )


def modal_command(*arguments):
    check_modal_credentials()
    if selected_modal_target is None:
        raise RuntimeError("Run Select workspace successfully before deploying, viewing logs or stopping.")
    profile, environment = selected_modal_target
    return ["modal", "--profile", profile, *arguments, "--env", environment]


check_modal_credentials()
subprocess.run(["modal", "profile", "list"], check=True)
profile = MODAL_PROFILE.strip() or input("Workspace/profile name from the list above: ").strip()
environment = MODAL_ENVIRONMENT.strip()
if not profile or not environment:
    raise ValueError("Enter both a workspace/profile name and an environment, then rerun this cell.")

# Read-only check verifies credentials and environment before any deployment.
# Explicit flags also keep subsequent commands on this target if the active profile changes.
subprocess.run(
    ["modal", "--profile", profile, "app", "list", "--env", environment],
    check=True,
)
selected_modal_target = (profile, environment)
print(f"\nSelected workspace/profile: {profile} | Environment: {environment}")
print("Deploy, logs and stop commands in this notebook will use this selection.")


In [ ]:
%%swhub_action write_app
# @title Register write app (runs only from the panel)
%%writefile modal_katago_contribute.py
"""KataGo v1.18 distributed-training contributor for SWHub on Modal.

Generated and deployed by Kata_Modal_Contribution.ipynb. To redeploy after
running this cell, use: modal deploy modal_katago_contribute.py

Then paste the printed wss:// URL plus /contribute into SWHub's
KataGo Contribution page.
"""

import asyncio
import hashlib
import hmac
import json
import os
import re
import shutil
import subprocess
import tempfile
import urllib.request
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import modal


APP_NAME = "Katago-Contribute"
MODAL_GPU = "T4"
SESSION_TIMEOUT_HOURS = 4

KATAGO_RELEASE = "v1.18.2"
KATAGO_URL = (
    "https://github.com/lightvector/KataGo/releases/download/v1.18.2/"
    "katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip"
)
KATAGO_SHA256 = "2cb197fc9bc2050f29112abb5b83cdbad63936cb4605d6e199772aedc9806c02"

CACHE = Path("/cache")
KATAGO_DIR = CACHE / "katago_v1_18_2_cuda12_8_cudnn9"
KATAGO_ARCHIVE = KATAGO_DIR / "katago.zip"
KATAGO_BIN = KATAGO_DIR / "katago"
KATAGO_EXTRACTED_BIN = KATAGO_DIR / "squashfs-root" / "AppRun"
CONTRIBUTE_BASE_DIR = CACHE / "contribute"
HOME_DATA_DIR = CACHE / "home"
LOG_DIR = CACHE / "logs"

app = modal.App(APP_NAME)
volume = modal.Volume.from_name("katago-contribute", create_if_missing=True)
if modal.is_local():
    endpoint_auth_secret = modal.Secret.from_dict(
        {
            "SWHUB_CONTRIBUTOR_USERNAME": os.environ.get(
                "SWHUB_CONTRIBUTOR_USERNAME", ""
            ),
            "SWHUB_CONTRIBUTOR_PASSWORD": os.environ.get(
                "SWHUB_CONTRIBUTOR_PASSWORD", ""
            ),
        }
    )
else:
    endpoint_auth_secret = modal.Secret.from_dict({})
image = (
    modal.Image.from_registry(
        "nvidia/cuda:12.8.1-cudnn-runtime-ubuntu22.04",
        add_python="3.11",
    )
    .entrypoint([])
    .apt_install(
        "ca-certificates",
        "curl",
        "libgomp1",
        "libomp-dev",
        "libzip4",
        "p7zip-full",
        "unzip",
        "wget",
    )
    .pip_install("fastapi[standard]==0.162.2", "uvicorn==0.54.0", "websockets==17.4")
)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _download(url: str, target: Path, expected_sha256: str) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        if _sha256(target) == expected_sha256:
            return
        target.unlink()
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "SWHub-KataGo-Modal-Contributor/1.0",
            "Accept": "*/*",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
    except Exception:
        temporary.unlink(missing_ok=True)
        subprocess.run(
            [
                "wget",
                "--tries=3",
                "--timeout=60",
                "--user-agent=SWHub-KataGo-Modal-Contributor/1.0",
                "-O",
                str(temporary),
                url,
            ],
            check=True,
        )
    if not temporary.exists() or temporary.stat().st_size == 0:
        raise RuntimeError(f"Download produced an empty file: {url}")
    actual_sha256 = _sha256(temporary)
    if actual_sha256 != expected_sha256:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Download checksum mismatch for {url}: {actual_sha256}"
        )
    temporary.replace(target)


def _command_output(
    arguments: list[str],
    cwd: Path | None = None,
    timeout: int = 120,
) -> tuple[int, str]:
    result = subprocess.run(
        arguments,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
        timeout=timeout,
    )
    return result.returncode, result.stdout


def ensure_katago() -> Path:
    KATAGO_DIR.mkdir(parents=True, exist_ok=True)
    if not KATAGO_BIN.exists():
        _download(KATAGO_URL, KATAGO_ARCHIVE, KATAGO_SHA256)
        with zipfile.ZipFile(KATAGO_ARCHIVE) as archive:
            archive.extractall(KATAGO_DIR)
        KATAGO_ARCHIVE.unlink(missing_ok=True)
    if not KATAGO_BIN.exists():
        raise RuntimeError(f"KataGo archive did not contain {KATAGO_BIN}")
    KATAGO_BIN.chmod(0o755)

    code, output = _command_output([str(KATAGO_BIN), "version"])
    if code == 0 and KATAGO_RELEASE.removeprefix("v") in output:
        return KATAGO_BIN

    if not KATAGO_EXTRACTED_BIN.exists():
        code, extract_output = _command_output(
            [str(KATAGO_BIN), "--appimage-extract"],
            cwd=KATAGO_DIR,
        )
        if code != 0:
            raise RuntimeError(
                "KataGo AppImage extraction failed:\n" + extract_output
            )
    # Restored caches can lose permissions on the bundled executable too.
    (KATAGO_EXTRACTED_BIN.parent / "usr" / "bin" / "katago").chmod(0o755)
    KATAGO_EXTRACTED_BIN.chmod(0o755)
    code, output = _command_output([str(KATAGO_EXTRACTED_BIN), "version"])
    if code != 0 or KATAGO_RELEASE.removeprefix("v") not in output:
        raise RuntimeError("KataGo v1.18 validation failed:\n" + output)
    return KATAGO_EXTRACTED_BIN


def _bool(value: object) -> str:
    return "true" if value is True else "false"


def _quoted(value: object) -> str:
    return json.dumps(str(value), ensure_ascii=False)


def _int_in_range(
    config: dict,
    key: str,
    default: int,
    minimum: int,
    maximum: int,
) -> int:
    try:
        value = int(config.get(key, default))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{key} must be a number") from exc
    if value < minimum or value > maximum:
        raise ValueError(f"{key} must be between {minimum} and {maximum}")
    return value


def _tri_state(config: dict, key: str) -> str:
    value = str(config.get(key, "auto")).lower()
    if value not in {"auto", "true", "false"}:
        raise ValueError(f"{key} must be auto, true, or false")
    return value


_PROTECTED_EXTRA_KEYS = {
    "cudadevicetouse",
    "cudausefp16",
    "cudausenhwc",
    "disablepredownloadloop",
    "homedatadir",
    "includeownership",
    "logdir",
    "loggamesasjson",
    "maxratingmatches",
    "maxsimultaneousgames",
    "mirroruseproxy",
    "modeldownloadmirrorbaseurl",
    "numnnserverthreadspermodel",
    "onlyplayratingmatches",
    "password",
    "proxybasicauthpassword",
    "proxybasicauthusername",
    "proxyhost",
    "proxyport",
    "reportperformanceevery",
    "serverurl",
    "taskrepfactor",
    "username",
    "warntaskunusedkeys",
    "watchongoinggameinfile",
    "watchongoinggameinfilename",
}


def _safe_extra_lines(raw: object) -> list[str]:
    raw_text = str(raw or "")
    if len(raw_text) > 16000:
        raise ValueError("Additional config must be 16,000 characters or fewer")
    lines: list[str] = []
    for line in raw_text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            lines.append(line)
            continue
        match = re.match(r"^([A-Za-z][A-Za-z0-9_]*)\s*=", stripped)
        if not match:
            raise ValueError(
                f"Invalid additional config line (expected key = value): {line}"
            )
        if match.group(1).lower() in _PROTECTED_EXTRA_KEYS:
            raise ValueError(
                f"{match.group(1)} is managed by SWHub and cannot be overridden"
            )
        lines.append(line)
    return lines


def build_config(config: dict) -> str:
    server_url = str(config.get("serverUrl", "")).strip()
    parsed_server_url = urlparse(server_url)
    if (
        parsed_server_url.scheme != "https"
        or not parsed_server_url.netloc
        or parsed_server_url.username is not None
        or parsed_server_url.password is not None
    ):
        raise ValueError("serverUrl must be a valid https:// URL")
    username = str(config.get("username", "")).strip()
    password = str(config.get("password", ""))
    if not username or not password:
        raise ValueError("username and password are required")

    proxy_host = str(config.get("proxyHost", "")).strip()
    proxy_username = str(config.get("proxyBasicAuthUsername", "")).strip()
    proxy_password = str(config.get("proxyBasicAuthPassword", ""))
    mirror_url = str(config.get("modelDownloadMirrorBaseUrl", "")).strip()
    if mirror_url:
        parsed_mirror_url = urlparse(mirror_url)
        if (
            parsed_mirror_url.scheme != "https"
            or not parsed_mirror_url.netloc
            or parsed_mirror_url.username is not None
            or parsed_mirror_url.password is not None
        ):
            raise ValueError("modelDownloadMirrorBaseUrl must be https://")
    watch_name = Path(
        str(config.get("watchOngoingGameInFileName") or "watchgame.txt")
    ).name
    max_games = _int_in_range(
        config, "maxSimultaneousGames", 8, 1, 128
    )
    max_rating = _int_in_range(config, "maxRatingMatches", 1, 0, 128)
    task_rep_factor = _int_in_range(config, "taskRepFactor", 4, 2, 16)
    report_performance_every = _int_in_range(
        config, "reportPerformanceEvery", 120, 1, 21600
    )
    nn_threads = _int_in_range(
        config, "numNNServerThreadsPerModel", 2, 1, 16
    )
    cuda_device = _int_in_range(config, "cudaDeviceToUse", 0, 0, 15)

    values = [
        f"serverUrl = {_quoted(server_url)}",
        f"username = {_quoted(username)}",
        f"password = {_quoted(password)}",
        f"maxSimultaneousGames = {max_games}",
        "logGamesAsJson = true",
        f"includeOwnership = {_bool(config.get('includeOwnership', False))}",
        "watchOngoingGameInFile = "
        + _bool(config.get("watchOngoingGameInFile", False)),
        f"watchOngoingGameInFileName = {_quoted(watch_name)}",
        "onlyPlayRatingMatches = "
        + _bool(config.get("onlyPlayRatingMatches", False)),
        f"maxRatingMatches = {max_rating}",
        "disablePredownloadLoop = "
        + _bool(config.get("disablePredownloadLoop", False)),
        f"taskRepFactor = {task_rep_factor}",
        f"reportPerformanceEvery = {report_performance_every}",
        "warnTaskUnusedKeys = "
        + _bool(config.get("warnTaskUnusedKeys", False)),
        f"numNNServerThreadsPerModel = {nn_threads}",
        f"cudaDeviceToUse = {cuda_device}",
        f"cudaUseFP16 = {_tri_state(config, 'cudaUseFP16')}",
        f"cudaUseNHWC = {_tri_state(config, 'cudaUseNHWC')}",
        f"homeDataDir = {_quoted(HOME_DATA_DIR)}",
        f"logDir = {_quoted(LOG_DIR)}",
    ]
    if proxy_host:
        proxy_port = _int_in_range(config, "proxyPort", 0, 1, 65535)
        values.extend(
            [
                f"proxyHost = {_quoted(proxy_host)}",
                f"proxyPort = {proxy_port}",
            ]
        )
        if proxy_username:
            values.extend(
                [
                    f"proxyBasicAuthUsername = {_quoted(proxy_username)}",
                    f"proxyBasicAuthPassword = {_quoted(proxy_password)}",
                ]
            )
    if mirror_url:
        values.extend(
            [
                f"modelDownloadMirrorBaseUrl = {_quoted(mirror_url)}",
                "mirrorUseProxy = "
                + _bool(config.get("mirrorUseProxy", True)),
            ]
        )
    values.extend(_safe_extra_lines(config.get("extraConfig")))
    return "\n".join(values) + "\n"


@app.function(
    image=image,
    gpu=MODAL_GPU,
    cpu=4,
    memory=8192,
    timeout=SESSION_TIMEOUT_HOURS * 60 * 60,
    startup_timeout=20 * 60,
    max_containers=1,
    scaledown_window=60,
    volumes={"/cache": volume},
    secrets=[endpoint_auth_secret],
)
@modal.concurrent(max_inputs=1)
@modal.asgi_app()
def gg():
    from fastapi import FastAPI, WebSocket, WebSocketDisconnect

    web_app = FastAPI()

    @web_app.get("/")
    async def status():
        return {
            "ok": True,
            "app": APP_NAME,
            "katago": KATAGO_RELEASE,
            "gpu": MODAL_GPU,
            "protocol": "swhub-katago-contribution",
            "protocolVersion": 1,
            "websocket": "/contribute",
            "note": "Connect with the SWHub KataGo Contribution page.",
        }

    @web_app.websocket("/contribute")
    async def contribute(websocket: WebSocket):
        await websocket.accept()
        process: asyncio.subprocess.Process | None = None
        config_path: Path | None = None
        reader_tasks: list[asyncio.Task] = []
        uploaded_games = 0
        latest_games: dict[str, dict] = {}
        send_lock = asyncio.Lock()
        shutting_down = False
        paused = False
        sensitive_values: list[str] = []

        def redact(text: str) -> str:
            result = text
            for value in sensitive_values:
                if value:
                    result = result.replace(value, "[REDACTED]")
            return result

        async def send(event_type: str, **data) -> None:
            payload = {"type": event_type, **data}
            async with send_lock:
                await websocket.send_text(
                    json.dumps(payload, separators=(",", ":"))
                )

        async def write_command(command: str) -> None:
            if process is None or process.stdin is None:
                return
            process.stdin.write((command + "\n").encode())
            await process.stdin.drain()

        def close_input_after_cleanup(text: str) -> None:
            # KataGo leaves its control thread blocked in getline(cin) at exit.
            # EOF releases it so C++ stream cleanup cannot hang after games and
            # uploads are done. Keep stdin open until this final log so forcequit
            # remains available throughout graceful shutdown.
            if text.endswith("All cleaned up, quitting"):
                if process is not None and process.stdin is not None:
                    process.stdin.close()

        async def commit_cache() -> None:
            try:
                await asyncio.to_thread(volume.commit)
            except Exception as exc:
                message = f"Volume commit warning: {exc!r}"
                print(message, flush=True)
                try:
                    await send("log", message=message)
                except Exception:
                    pass

        async def periodic_commit() -> None:
            while process is not None and process.returncode is None:
                await asyncio.sleep(10 * 60)
                await commit_cache()

        def finish_data_from_upload_log(
            log_line: str,
        ) -> tuple[str, dict] | None:
            match = re.search(
                r"\buploaded sgf\s+(\S+\.sgf)(?:\s|$)",
                log_line,
                flags=re.IGNORECASE,
            )
            if match is None:
                return None
            sgf_path = Path(match.group(1))
            finish_data: dict[str, object] = {"finished": True}
            try:
                sgf_text = sgf_path.read_text(
                    encoding="utf-8", errors="replace"
                )
            except OSError:
                return sgf_path.stem, finish_data
            result_match = re.search(r"RE\[([^\]]*)\]", sgf_text)
            if result_match is None:
                return sgf_path.stem, finish_data
            result = result_match.group(1).strip()
            if result:
                finish_data["result"] = result
                upper_result = result.upper()
                if upper_result.startswith("B+"):
                    finish_data["winner"] = "B"
                elif upper_result.startswith("W+"):
                    finish_data["winner"] = "W"
                finish_data["resigned"] = upper_result.endswith("+R")
            return sgf_path.stem, finish_data

        async def read_stdout() -> None:
            nonlocal uploaded_games
            assert process is not None and process.stdout is not None
            while True:
                line = await process.stdout.readline()
                if not line:
                    return
                raw_text = line.decode(errors="ignore").strip()
                close_input_after_cleanup(raw_text)
                text = redact(raw_text)
                if not text:
                    continue
                if text.startswith("{"):
                    try:
                        data = json.loads(text)
                        if isinstance(data, dict) and data.get("gameId"):
                            latest_games[str(data["gameId"])] = dict(data)
                            await send("game", data=data)
                            continue
                    except json.JSONDecodeError:
                        pass
                if "uploaded sgf" in text.lower():
                    finish = await asyncio.to_thread(
                        finish_data_from_upload_log, raw_text
                    )
                    if finish is not None:
                        game_id, finish_data = finish
                        latest = latest_games.pop(game_id, None)
                        if latest is not None:
                            finished = {**latest, **finish_data}
                            await send("game", data=finished)
                    uploaded_games += 1
                    await send(
                        "uploaded",
                        uploadedGames=uploaded_games,
                        message=(
                            f"Uploaded game {uploaded_games}"
                            + (f" · {finish[0]}" if finish is not None else "")
                        ),
                    )
                else:
                    await send("log", stream="stdout", message=text)

        async def read_stderr() -> None:
            assert process is not None and process.stderr is not None
            while True:
                line = await process.stderr.readline()
                if not line:
                    return
                raw_text = line.decode(errors="ignore").strip()
                close_input_after_cleanup(raw_text)
                text = redact(raw_text)
                if not text:
                    continue
                lower = text.lower()
                event_type = (
                    "error"
                    if "uncaught exception" in lower
                    or "server returned error" in lower
                    or "not status code 200 ok" in lower
                    else "log"
                )
                await send(event_type, stream="stderr", message=text)

        async def drain_output_readers() -> None:
            output_readers = reader_tasks[:2]
            if not output_readers:
                return
            _, pending = await asyncio.wait(output_readers, timeout=10)
            for task in pending:
                task.cancel()
            await asyncio.gather(*output_readers, return_exceptions=True)

        async def stop_process(force: bool) -> None:
            nonlocal shutting_down, paused
            if process is None or process.returncode is not None:
                return
            shutting_down = True
            try:
                if not force and paused:
                    await write_command("resume")
                    paused = False
                await write_command("forcequit" if force else "quit")
            except (BrokenPipeError, ConnectionResetError):
                pass
            try:
                await asyncio.wait_for(
                    process.wait(),
                    timeout=15 if force else 120,
                )
            except asyncio.TimeoutError:
                try:
                    process.terminate()
                except ProcessLookupError:
                    return
                try:
                    await asyncio.wait_for(process.wait(), timeout=10)
                except asyncio.TimeoutError:
                    try:
                        process.kill()
                    except ProcessLookupError:
                        return
                    await process.wait()

        try:
            await send(
                "hello",
                protocol="swhub-katago-contribution",
                protocolVersion=1,
                message=f"Modal {MODAL_GPU} contributor ready for KataGo {KATAGO_RELEASE}.",
            )
            raw = await asyncio.wait_for(websocket.receive_text(), timeout=60)
            try:
                request = json.loads(raw)
            except json.JSONDecodeError:
                await send("error", message="Start request must be valid JSON.")
                await websocket.close(code=4400)
                return
            if not isinstance(request, dict):
                await send("error", message="Start request must be a JSON object.")
                await websocket.close(code=4400)
                return
            if request.get("protocolVersion") != 1:
                await send(
                    "error",
                    message="Unsupported SWHub contribution protocol version.",
                )
                await websocket.close(code=4400)
                return
            expected_endpoint_username = os.environ.get(
                "SWHUB_CONTRIBUTOR_USERNAME", ""
            )
            expected_endpoint_password = os.environ.get(
                "SWHUB_CONTRIBUTOR_PASSWORD", ""
            )
            endpoint_auth = request.get("endpointAuth")
            if bool(expected_endpoint_username) != bool(expected_endpoint_password):
                await send(
                    "error",
                    message="Modal endpoint login is configured incompletely.",
                )
                await websocket.close(code=1011)
                return
            if expected_endpoint_username:
                endpoint_username = (
                    str(endpoint_auth.get("username", ""))
                    if isinstance(endpoint_auth, dict)
                    else ""
                )
                endpoint_password = (
                    str(endpoint_auth.get("password", ""))
                    if isinstance(endpoint_auth, dict)
                    else ""
                )
                if not (
                    hmac.compare_digest(
                        endpoint_username, expected_endpoint_username
                    )
                    and hmac.compare_digest(
                        endpoint_password, expected_endpoint_password
                    )
                ):
                    await send("error", message="Contributor login failed.")
                    await websocket.close(code=4401)
                    return
            if request.get("action") != "start" or not isinstance(
                request.get("config"), dict
            ):
                await send("error", message="First message must be a start request.")
                await websocket.close(code=4400)
                return
            sensitive_values.extend(
                [
                    str(request["config"].get("password", "")),
                    str(request["config"].get("proxyBasicAuthPassword", "")),
                    str(endpoint_auth.get("password", ""))
                    if isinstance(endpoint_auth, dict)
                    else "",
                ]
            )

            await send("connecting", message="Preparing KataGo v1.18.2 in Modal, mandatory contribution safety checks run before games begin…")
            katago = await asyncio.to_thread(ensure_katago)
            contribution_dirs = [
                CONTRIBUTE_BASE_DIR,
                HOME_DATA_DIR,
                LOG_DIR,
            ]
            for directory in contribution_dirs:
                directory.mkdir(parents=True, exist_ok=True)

            config_text = build_config(request["config"])
            descriptor, temporary_name = tempfile.mkstemp(
                prefix="swhub_contribute_",
                suffix=".cfg",
                dir="/tmp",
                text=True,
            )
            config_path = Path(temporary_name)
            with os.fdopen(descriptor, "w", encoding="utf-8") as output:
                output.write(config_text)
            config_path.chmod(0o600)

            arguments = [
                str(katago),
                "contribute",
                "-config",
                str(config_path),
                "-base-dir",
                str(CONTRIBUTE_BASE_DIR),
            ]
            process = await asyncio.create_subprocess_exec(
                *arguments,
                stdin=asyncio.subprocess.PIPE,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
                limit=50 * 1024 * 1024,
            )
            await send(
                "started",
                message=(
                    f"KataGo {KATAGO_RELEASE} is contributing on Modal {MODAL_GPU}."
                ),
            )
            reader_tasks = [
                asyncio.create_task(read_stdout()),
                asyncio.create_task(read_stderr()),
                asyncio.create_task(periodic_commit()),
            ]

            process_wait = asyncio.create_task(process.wait())
            while process.returncode is None:
                receive = asyncio.create_task(websocket.receive_text())
                done, _ = await asyncio.wait(
                    {receive, process_wait},
                    return_when=asyncio.FIRST_COMPLETED,
                )
                if process_wait in done:
                    receive.cancel()
                    await asyncio.gather(receive, return_exceptions=True)
                    break
                raw = receive.result()
                try:
                    command_message = json.loads(raw)
                except json.JSONDecodeError:
                    await send("warning", message="Command must be valid JSON.")
                    continue
                if not isinstance(command_message, dict):
                    await send(
                        "warning",
                        message="Command must be a JSON object.",
                    )
                    continue
                command = command_message.get("action")
                if command == "pause":
                    await write_command("pause")
                    paused = True
                    await send("paused", paused=True, message="Contribution paused.")
                elif command == "resume":
                    await write_command("resume")
                    paused = False
                    await send(
                        "resumed", paused=False, message="Contribution resumed."
                    )
                elif command == "quit":
                    shutting_down = True
                    if paused:
                        await write_command("resume")
                        paused = False
                    await write_command("quit")
                    await send(
                        "stopping",
                        message=(
                            "KataGo is finishing active games and will not start "
                            "new ones."
                        ),
                    )
                elif command == "forcequit":
                    await send("stopping", message="Force-stopping KataGo…")
                    await stop_process(force=True)
                    break
                else:
                    await send(
                        "warning",
                        message=f"Unsupported contribution command: {command!r}",
                    )

            return_code = await process_wait
            await drain_output_readers()
            await send(
                "stopped",
                uploadedGames=uploaded_games,
                message=f"KataGo exited with code {return_code}.",
            )
        except WebSocketDisconnect:
            print("SWHub viewer disconnected, stopping KataGo.", flush=True)
        except asyncio.TimeoutError:
            try:
                await send("error", message="Timed out waiting for startup data.")
            except Exception:
                pass
        except Exception as exc:
            try:
                await send("error", message=f"Contributor failed: {exc}")
            except Exception:
                pass
            print(f"Contributor failed: {exc!r}", flush=True)
        finally:
            if process is not None and process.returncode is None:
                await stop_process(force=True if not shutting_down else False)
            for task in reader_tasks:
                task.cancel()
            if reader_tasks:
                await asyncio.gather(*reader_tasks, return_exceptions=True)
            if config_path is not None:
                config_path.unlink(missing_ok=True)
            if process is not None:
                await commit_cache()

    return web_app


In [ ]:
%%swhub_action deploy
# @title Register deploy (runs only from the panel)
import os, re, subprocess
from pathlib import Path

if "modal_command" not in globals():
    raise RuntimeError("Click Select workspace before deploying.")

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"
if globals().get("ENABLE_CONTRIBUTOR_LOGIN", False):
    username = globals().get("CONTRIBUTOR_LOGIN_USERNAME", "").strip()
    password = globals().get("CONTRIBUTOR_LOGIN_PASSWORD", "")
    if not username or not password:
        raise ValueError("Fill the Controller login fields again before deploying.")
    env["SWHUB_CONTRIBUTOR_USERNAME"] = username
    env["SWHUB_CONTRIBUTOR_PASSWORD"] = password

script_path = Path.cwd() / "modal_katago_contribute.py"
if not script_path.exists():
    raise RuntimeError("Click Deploy to prepare the server app before deployment.")

deploy = subprocess.Popen(
    modal_command("deploy", str(script_path)),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
deploy_lines = []
for line in deploy.stdout:
    print(line, end="")
    deploy_lines.append(line)
if deploy.wait() != 0:
    raise RuntimeError("Modal deployment failed")

urls = []
for line in deploy_lines:
    for url in re.findall(r"https://[^\s]+?\.modal\.run", line):
        clean = url.rstrip(".,)];'")
        if clean not in urls:
            urls.append(clean)

if urls:
    base_url = urls[-1].rstrip("/")
    websocket_url = base_url.replace("https://", "wss://") + "/contribute"
    print("\nPaste this into SWHub's Contributor WebSocket endpoint field:")
    print(websocket_url)
else:
    print("\nCopy the https://...modal.run URL above, change https to wss, and add /contribute.")


In [ ]:
%%swhub_action stop
# @title Register stop (runs only from the panel)
import subprocess
# subprocess.run(modal_command("app", "logs", "Katago-Contribute", "--follow"), check=True)
subprocess.run(modal_command("app", "stop", "Katago-Contribute"), check=True)


## Troubleshooting

| What happened? | What to do |
| --- | --- |
| Modal asks for a payment method | Add a valid payment method in Modal's billing settings. GPU use requires one even with free credits. |
| Your workspace is missing or incorrect | Click Login again, choose the intended workspace in the browser, then click Select workspace. |
| Login expired | Click Login and Select workspace again, then Deploy. |
| Credits remain but the service stopped | Check the workspace usage budget and unlocked credits in Modal's dashboard. |
| The GPU is unavailable | Retry later. This notebook requests a T4. |
| Colab restarted | Run the top Open control panel block again, then use Login and Select workspace before deploying again. Your deployed app is still on Modal unless it was stopped. |
| Deployment is blocked by an authentication rule | Ask the workspace owner about the rule. This setup needs an environment that permits its connection method. Adding the contributor's optional password does not satisfy Modal's separate proxy-authentication rule. |

<details>
<summary>Optional: recover from a Colab runtime update</summary>

Use **Runtime > Change runtime type > Runtime Version** to try an available earlier version, such as `2026.07`, then rerun setup. Keep a CPU runtime for these Modal setup notebooks. Older versions are temporary options. [Google's runtime help](https://research.google.com/colaboratory/runtime-version-faq.html)

</details>

If contribution stops after four hours, start a new session from SWHub. Use Finish games & stop to upload in-progress games before ending a session.
